In [1]:
import paths, dataframes

In [2]:
from pipe import select
import pandas

In [3]:
every_hive_audio_features = pandas.concat(
    paths.all_audio_features_csv_filepaths(hive_numbers=[1,2,3,4,5,6])
    | select(dataframes.from_filepath),
    ignore_index=True,
)

In [4]:
every_hive_audio_features.drop(columns=[c for c in every_hive_audio_features.columns if c.startswith("Unnamed")], inplace=True)
every_hive_audio_features.head()

,timestamp,hive,time_slice,spectral_centroid_mean,spectral_centroid_std,spectral_bandwidth_mean,spectral_bandwidth_std,spectral_rolloff_mean,spectral_rolloff_std,spectral_flatness_mean,...,low_to_high_energy_ratio_mean,low_to_high_energy_ratio_std,middle_to_high_energy_ratio_mean,middle_to_high_energy_ratio_std,low_modulation_energy,low_peak_modulation_frequency,middle_modulation_energy,middle_peak_modulation_frequency,high_modulation_energy,high_peak_modulation_frequency
0,2026-03-07 17:00:00,hive_01,17-18,1704.839665,191.237211,2283.109757,131.021388,4787.536582,572.251570,0.020389,...,2.975306,1.378878,2.208398,0.333739,3.415686e+06,2.031807,3.818026e+06,1.288793,2.246383e+06,1.878730
1,2026-03-07 18:00:00,hive_01,18-19,1790.814496,309.629397,2327.348104,179.594982,5004.947215,650.518700,0.021843,...,3.089824,0.908164,2.136371,0.167763,3.277247e+06,2.361399,3.380961e+06,5.802804,2.044457e+06,1.063060
2,2026-03-07 19:00:00,hive_01,19-20,1823.045205,315.363740,2333.649956,171.735514,5072.253507,625.039905,0.024116,...,2.926316,0.791499,2.195237,0.156738,2.871789e+06,1.005282,3.223628e+06,4.194463,1.880883e+06,5.745026
3,2026-03-07 20:00:00,hive_01,20-21,1805.283076,191.214024,2347.317531,115.439600,5128.420892,422.853036,0.022318,...,3.067312,0.931899,2.241347,0.177238,2.812510e+06,1.143894,3.075437e+06,2.307232,1.774921e+06,3.222514
4,2026-03-07 21:00:00,hive_01,21-22,1803.259381,147.803181,2313.412297,97.664542,5032.523823,377.836613,0.025464,...,2.758642,0.670456,2.205859,0.162945,2.481559e+06,3.766128,2.905298e+06,7.159476,1.691767e+06,10.150601


In [5]:
every_hive_audio_features["queenlessness"] = every_hive_audio_features.apply(dataframes.queenstate_from_row, axis=1) == "queenless"

In [6]:
ambient_features = pandas.read_csv(
    paths.ambient_features_filepath, parse_dates=["timestamp"]
)

In [7]:
ambient_features.drop(columns=[c for c in ambient_features.columns if c.startswith("Unnamed")], inplace=True)

In [8]:
merged_features_dataframe = every_hive_audio_features.merge(ambient_features, on="timestamp", how="left")

In [9]:
dataframes.saved_csv_filepath_from_features_dataframe(
    dataframe=merged_features_dataframe,
    dataframe_filename=paths.all_merged_features_filename,
)

'../../data/features/all_merged_features.csv'

In [10]:
import os
assert os.path.isfile(paths.all_merged_features_filepath)

In [11]:
# which columns have NaN, and how many
merged_features_dataframe.isnull().sum()[merged_features_dataframe.isnull().sum() > 0]

temperature    108
humidity       108
dtype: int64

In [13]:
hive_04_audio = every_hive_audio_features[every_hive_audio_features["hive"] == "hive_04"]
hive_04_audio[hive_04_audio["timestamp"].duplicated(keep=False)].sort_values("timestamp")

,timestamp,hive,time_slice,spectral_centroid_mean,spectral_centroid_std,spectral_bandwidth_mean,spectral_bandwidth_std,spectral_rolloff_mean,spectral_rolloff_std,spectral_flatness_mean,...,low_to_high_energy_ratio_std,middle_to_high_energy_ratio_mean,middle_to_high_energy_ratio_std,low_modulation_energy,low_peak_modulation_frequency,middle_modulation_energy,middle_peak_modulation_frequency,high_modulation_energy,high_peak_modulation_frequency,queenlessness
1283,2026-03-11 11:00:00,hive_04,11-12,1361.754858,1142.656932,1499.580965,1245.375309,3518.418246,2934.289824,0.446580,...,0.789353,0.714995,0.604438,2.842446e+06,1.030410,6.480053e+06,1.036192,5.707569e+06,1.451183,True
1285,2026-03-11 11:00:00,hive_04,11-12,775.198796,1181.456575,814.764588,1236.115885,1951.854581,2963.157288,0.720049,...,0.474854,0.378087,0.582956,4.155778e+05,1.002254,1.300785e+06,1.129987,1.101111e+06,3.330017,True
1287,2026-03-11 11:00:00,hive_04,11-12,2282.921089,203.544916,2520.380263,105.067509,5909.138257,388.368582,0.066037,...,0.760742,1.228266,0.127469,4.047706e+05,1.028347,9.788574e+05,7.227617,8.446658e+05,7.691146,True
1284,2026-03-11 12:00:00,hive_04,12-13,863.689537,1119.880505,962.191774,1239.871892,2259.351261,2918.295065,0.646637,...,0.758243,0.455454,0.591799,2.116200e+06,19.387571,4.287508e+06,1.335988,3.721193e+06,1.053015,True
1286,2026-03-11 12:00:00,hive_04,12-13,430.867390,901.747778,478.662939,997.542858,1127.799554,2354.822838,0.824750,...,0.382652,0.222366,0.466073,2.774778e+06,1.315317,7.879411e+06,1.278335,6.616982e+06,1.042481,True
1288,2026-03-11 12:00:00,hive_04,12-13,2325.607367,212.712250,2541.380138,100.619191,6034.220374,368.683088,0.071880,...,0.517601,1.208504,0.191204,1.377854e+06,1.158944,3.386463e+06,3.590590,3.016859e+06,2.085031,True


In [17]:
hive_04_csv = paths.features_folderpath + paths.audio_features_filename_from_hive_number(4)
pandas.read_csv(hive_04_csv).query("timestamp == '2026-03-11 11:00:00'")

,Unnamed: 0,timestamp,hive,time_slice,spectral_centroid_mean,spectral_centroid_std,spectral_bandwidth_mean,spectral_bandwidth_std,spectral_rolloff_mean,spectral_rolloff_std,...,low_to_high_energy_ratio_mean,low_to_high_energy_ratio_std,middle_to_high_energy_ratio_mean,middle_to_high_energy_ratio_std,low_modulation_energy,low_peak_modulation_frequency,middle_modulation_energy,middle_peak_modulation_frequency,high_modulation_energy,high_peak_modulation_frequency
58,58,2026-03-11 11:00:00,hive_04,11-12,1361.754858,1142.656932,1499.580965,1245.375309,3518.418246,2934.289824,...,0.553429,0.789353,0.714995,0.604438,2.842446e+06,1.030410,6.480053e+06,1.036192,5.707569e+06,1.451183
60,60,2026-03-11 11:00:00,hive_04,11-12,775.198796,1181.456575,814.764588,1236.115885,1951.854581,2963.157288,...,0.252212,0.474854,0.378087,0.582956,4.155778e+05,1.002254,1.300785e+06,1.129987,1.101111e+06,3.330017
62,62,2026-03-11 11:00:00,hive_04,11-12,2282.921089,203.544916,2520.380263,105.067509,5909.138257,388.368582,...,0.895219,0.760742,1.228266,0.127469,4.047706e+05,1.028347,9.788574e+05,7.227617,8.446658e+05,7.691146
